# পাঠ ০৯ - মেটাকগনিশন ডিজাইন প্যাটার্ন


## সেটআপ

এই নোটবুকটি মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে মেটাকগনিশন ডিজাইন প্যাটার্ন প্রদর্শন করে।

**প্রয়োজনীয়তাসমূহ:**
- পরিবেশ ভেরিয়েবলগুলির মাধ্যমে Azure OpenAI ডিপ্লয়মেন্ট কনফিগার করা হয়েছে
- Azure CLI মঞ্জুরিপ্রাপ্ত (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## মেটাকগনিশন কী?

মেটাকগনিশন হল **চিন্তার ওপর চিন্তা করা**। AI এজেন্টের প্রেক্ষিতে, এর অর্থ হল এমন এজেন্ট তৈরি করা যা পারে:

- নিজেদের আউটপুট এবং যুক্তি প্রক্রিয়ার ওপর **স্ব-প্রতিফলন** করা
- **ত্রুটি সনাক্ত** করা এবং নিঃশব্দে ব্যর্থ হওয়ার পরিবর্তে পরিশীলিতভাবে পুনরুদ্ধার করা
- তাদের প্রতিক্রিয়া পূর্ণাঙ্গ এবং সহায়ক কিনা তা **মূল্যায়ন** করা
- যখন প্রাথমিক পদ্ধতি কাজ না করে তখন তাদের কৌশল **পরিবর্তন** করা (যেমন, ব্যাকআপ সিস্টেমে ফিরে যাওয়া)

একটি মেটাকগনিটিভ এজেন্ট শুধু প্রশ্নের উত্তর দেয় না — এটি নিজের কর্মক্ষমতা মনিটর করে এবং ঝটপটে সমন্বয় করে।


## প্রাথমিক এবং ব্যাকআপ সরঞ্জাম

একটি সাধারণ মেটাকগনিশন প্যাটার্ন হল **ফলব্যাক স্ট্র্যাটেজি**। এজেন্ট প্রথমে একটি প্রাথমিক সরঞ্জাম চেষ্টা করে; এটি ব্যর্থ হলে (যেমন, 404 ত্রুটি), এজেন্ট ব্যর্থতা চিনতে পারে এবং স্বচ্ছন্দভাবে একটি ব্যাকআপ সরঞ্জামে স্যুইচ করে।

এটি বাস্তব বিশ্বের সিস্টেমগুলির অনুরূপ, যেখানে প্রাথমিক সেবাগুলি উপলব্ধ নাও থাকতে পারে এবং এজেন্টদের বিকল্প পথ বেছে নেওয়ার আগে নিজে সমস্যা সনাক্ত করতে হয়।

নিচে আমরা দুটি ফ্লাইট লুকআপ সরঞ্জামের সংজ্ঞা দিচ্ছি:
- **প্রাথমিক** — প্যারিস, টোকিও, এবং বার্সেলোনা অন্তর্ভুক্ত
- **ব্যাকআপ** — বার্লিন, সিডনি, এবং নিউ ইয়র্ক সিটি অন্তর্ভুক্ত


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ত্রুটি পুনরুদ্ধারসহ স্ব-প্রতিফলিত এজেন্ট

নিচের এজেন্টটিকে প্রথমে প্রাথমিক ফ্লাইট সিস্টেম চেষ্টা করতে, ব্যর্থতা সনাক্ত করতে এবং স্বচ্ছভাবে ব্যাকআপ সিস্টেমে ফিরে যেতে নির্দেশ দেওয়া হয়েছে। প্রতিটি প্রতিক্রিয়ার পরে এটি সংক্ষেপে নিজেই মূল্যায়ন করে যে এটি ব্যবহারকারীর প্রশ্নটি সম্পূর্ণরূপে উত্তর দিয়েছে কিনা।


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## স্ব-মূল্যায়ন প্যাটার্ন

মেটাকগনিশনের আরেকটি দিক হলো **স্ব-মূল্যায়ন**: একটি আলাদা এজেন্ট (অথবা একই এজেন্ট দ্বিতীয়বার) একটি প্রতিক্রিয়া পর্যালোচনা করে সম্পূর্ণতা, নির্ভুলতা, এবং সহায়কতা নিরূপণের জন্য।

নিচে আমরা একটি `ResponseEvaluator` এজেন্ট তৈরি করি যা ভ্রমণ এজেন্টের প্রতিক্রিয়াগুলোকে তিনটি মাত্রায় স্কোর করে।


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে **মেটাকগনিটিভ এজেন্ট** তৈরি করা যায়:

- **স্ব-মতামত**: এমন এজেন্ট যারা তাদের নিজস্ব যুক্তি পর্যবেক্ষণ করে এবং স্বচ্ছভাবে কি ঘটেছে তা যোগাযোগ করে।
- **ভুল পুনরুদ্ধার ফালব্যাক সহ**: একটি প্রাথমিক + ব্যাকআপ টুল প্যাটার্ন যেখানে এজেন্ট ব্যর্থতা (যেমন, 404 ত্রুটি) সনাক্ত করে এবং স্বয়ংক্রিয়ভাবে বিকল্প উৎস চেষ্টা করে।
- **স্ব-মূল্যায়ন**: একটি পৃথক মূল্যায়নকারী এজেন্ট যা পূর্ণতা, সঠিকতা, এবং সহায়কতার জন্য প্রতিক্রিয়াগুলি স্কোর করে।

এই প্যাটার্নগুলো এজেন্টদের আরও স্থিতিশীল, স্বচ্ছ, এবং বিশ্বাসযোগ্য করে তোলে — যা প্রোডাকশন স্থাপনের জন্য অত্যন্ত গুরুত্বপূর্ণ গুণাবলী।  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
